# EXAMEN GRUPAL — Semanas 9 y 10
## Modelado Secuencial con RNN, LSTM y GRU

**Curso:** Inteligencia Artificial aplicada a la Ingenieria  
**Docente:** M.Sc. Ing. Jairo Pinedo Taquia
**Modalidad:** Grupal — 4 integrantes  
**Duracion:** 2 horas  
**Escala de calificacion:** 0 a 20 puntos

---

### Datos del grupo

| Campo | Completar |
|-------|-----------|
| Integrante 1 (Datos) | |
| Integrante 2 (RNN) | |
| Integrante 3 (LSTM/GRU) | |
| Integrante 4 (Comunicacion) | |
| Seccion | |
| Fecha | |

---

### Descripcion del caso

Una planta industrial opera una bomba de agua cuyo comportamiento es registrado por 52 sensores de campo con frecuencia de un minuto durante varios meses. El sistema etiqueta cada registro con uno de tres estados operativos:

- `NORMAL`: operacion dentro de parametros
- `RECOVERING`: degradacion incipiente, señal de alerta
- `BROKEN`: falla confirmada

El area de confiabilidad necesita un modelo que anticipe el estado `RECOVERING` antes de que ocurra la falla, para programar intervenciones preventivas en lugar de correctivas.

El grupo actua como equipo de ingenieria de datos contratado para entregar un prototipo funcional y una presentacion ejecutiva.

**Dataset:** [https://www.kaggle.com/datasets/nphantawee/pump-sensor-data](https://www.kaggle.com/datasets/nphantawee/pump-sensor-data)  
Archivo: `sensor.csv`

---

### Estructura del examen y puntaje

| Fase | Contenido | Puntos | Tiempo estimado |
|------|-----------|--------|----------------|
| 1 | Exploracion y preparacion de datos | 4 | 20 min |
| 2 | Implementacion de modelos | 9 | 60 min |
| 3 | Analisis de gradientes (teoria) | 3 | 10 min |
| 4 | Presentacion ejecutiva | 4 | 10 min |
| **Total** | | **20** | **100 min** |

**Bonus:** Dashboard interactivo con Streamlit (+2 pts fuera del tiempo del examen)

---

### Instrucciones generales

1. Ejecutar las celdas en orden. No modificar las celdas marcadas como `[CELDA BASE — NO MODIFICAR]`.
2. Completar unicamente las celdas marcadas con `[COMPLETAR]`.
3. Las celdas Markdown de respuesta escrita deben editarse haciendo doble clic sobre ellas.
4. Al finalizar, compartir el notebook ejecutado via Google Drive con el docente.
5. Cualquier integrante puede ser preguntado sobre cualquier parte del notebook durante la exposicion oral.

---
## CELDA BASE — Configuracion e instalacion
### NO MODIFICAR

In [1]:
# ============================================================
# CELDA BASE — NO MODIFICAR
# Instalacion de dependencias y carga del dataset
# ============================================================

# Instalar kaggle si no esta disponible en el entorno
!pip install kaggle -q

import numpy as np                        # algebra lineal y operaciones numericas
import pandas as pd                       # manejo de DataFrames
import matplotlib.pyplot as plt           # visualizacion estatica
import matplotlib.patches as mpatches    # parches de color para leyendas manuales
import warnings                           # control de advertencias
warnings.filterwarnings('ignore')         # suprimir advertencias no criticas

from sklearn.preprocessing import StandardScaler, LabelEncoder   # normalizacion y codificacion
from sklearn.metrics import classification_report, confusion_matrix  # evaluacion del modelo
from sklearn.utils.class_weight import compute_class_weight          # pesos para clases desbalanceadas

import tensorflow as tf                                   # framework de deep learning
from tensorflow.keras.models import Sequential            # modelo secuencial de capas
from tensorflow.keras.layers import (                     # capas disponibles
    SimpleRNN, LSTM, GRU, Dense, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping      # detener entrenamiento si no mejora

# Configuracion de graficos
plt.rcParams['figure.dpi']       = 100
plt.rcParams['font.size']        = 10
plt.rcParams['axes.titlesize']   = 11
plt.rcParams['axes.titleweight'] = 'bold'

# Semilla global para reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")
print("Dependencias cargadas correctamente.")

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
!pip install

In [ ]:
# ============================================================
# CELDA BASE — NO MODIFICAR
# Carga del dataset desde Kaggle o subida manual
# ============================================================

# OPCION A: descarga automatica con API de Kaggle
# Requiere tener kaggle.json configurado en /root/.kaggle/
# Instrucciones: https://www.kaggle.com/docs/api
#
# import os
# from google.colab import files
# files.upload()   # subir kaggle.json cuando aparezca el dialogo
# os.makedirs('/root/.kaggle', exist_ok=True)
# os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
# os.chmod('/root/.kaggle/kaggle.json', 600)
# !kaggle datasets download -d nphantawee/pump-sensor-data --unzip

# OPCION B: subida manual (recomendada en examen)
# 1. Descargar sensor.csv desde: kaggle.com/datasets/nphantawee/pump-sensor-data
# 2. En Colab: panel izquierdo > icono de carpeta > subir archivo
# 3. Ejecutar la celda siguiente

try:
    df_raw = pd.read_csv('sensor.csv')                    # cargar el archivo CSV
    df_raw.columns = df_raw.columns.str.strip()           # eliminar espacios en nombres de columnas
    df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])  # convertir timestamp a datetime
    df_raw = df_raw.sort_values('timestamp').reset_index(drop=True)  # ordenar cronologicamente
    print(f"Dataset cargado: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
    print(f"Rango temporal: {df_raw['timestamp'].min()} a {df_raw['timestamp'].max()}")
    print(f"\nDistribucion de estados (machine_status):")
    print(df_raw['machine_status'].value_counts())
    print(f"\nPorcentaje de cada estado:")
    print((df_raw['machine_status'].value_counts(normalize=True)*100).round(2))
except FileNotFoundError:
    print("ERROR: archivo sensor.csv no encontrado.")
    print("Subir el archivo manualmente siguiendo las instrucciones de OPCION B.")

---
## FASE 1 — Exploracion y preparacion de datos
**Responsable principal:** Integrante 1  
**Puntaje:** 4 puntos  
**Tiempo estimado:** 20 minutos

### Rubrica

| Criterio | Puntos |
|----------|--------|
| Seleccion correcta de sensores (umbral de nulos aplicado) | 0.5 |
| Tratamiento de NaN con justificacion correcta | 0.5 |
| Grafico de series temporales con coloreado por estado | 1.0 |
| Deteccion y decision sobre la clase BROKEN | 1.0 |
| Justificacion escrita del modelado secuencial | 1.0 |
| **Total Fase 1** | **4.0** |

In [ ]:
# ============================================================
# FASE 1 — Tarea 1.1 [COMPLETAR]
# Seleccion de sensores y tratamiento de valores nulos
# ============================================================

# Paso 1: Identificar columnas de sensores (comienzan con 'sensor')
# Calcular el porcentaje de valores nulos de cada sensor
# Conservar solo los sensores con menos del 5% de nulos

# TODO: completar las siguientes lineas
# umbral_nulos = ...
# todos_sensores = [col for col in df_raw.columns if col.startswith('sensor')]
# sensores_validos = [...]
# print(f"Sensores totales: {len(todos_sensores)}")
# print(f"Sensores conservados (< {umbral_nulos*100:.0f}% nulos): {len(sensores_validos)}")

# Paso 2: Crear copia de trabajo con los sensores seleccionados
# df = df_raw[['timestamp', 'machine_status'] + sensores_validos].copy()

# Paso 3: Imputar NaN con forward fill seguido de backward fill
# Justificacion: en series temporales de sensores, propagar el ultimo
# valor conocido es mejor estimador que la media global, ya que
# preserva la tendencia local de la señal
# df[sensores_validos] = df[sensores_validos].fillna(method='ffill').fillna(method='bfill')

# Paso 4: Verificar que no quedan nulos
# print(f"\nNulos restantes: {df[sensores_validos].isna().sum().sum()}")

print("[COMPLETAR] Implementar los pasos indicados en los comentarios.")

In [ ]:
# ============================================================
# FASE 1 — Tarea 1.2 [COMPLETAR]
# Visualizacion de series temporales por estado operativo
# ============================================================

# Graficar 4 sensores representativos en funcion del tiempo
# Colorear el fondo segun machine_status:
#   NORMAL     -> azul claro
#   RECOVERING -> naranja
#   BROKEN     -> rojo

# Guia de implementacion:
# 1. Elegir 4 sensores de la lista sensores_validos
# 2. Para cada sensor, crear un subplot
# 3. Usar ax.axvspan() para colorear los intervalos de cada estado
#    Ejemplo: ax.axvspan(t_inicio, t_fin, alpha=0.2, color='orange')
# 4. Etiquetar ejes con unidades y agregar leyenda de estados

# SENSORES_VIZ = sensores_validos[:4]   # seleccionar los primeros 4 como punto de partida

# TODO: implementar el grafico

print("[COMPLETAR] Implementar la visualizacion de series temporales.")

In [ ]:
# ============================================================
# FASE 1 — Tarea 1.3 [COMPLETAR]
# Decision sobre la clase BROKEN y codificacion de etiquetas
# ============================================================

# Paso 1: Contar cuantos registros tiene cada clase
# print(df['machine_status'].value_counts())

# Paso 2: El grupo debe tomar una decision sobre BROKEN
# Opciones:
#   A) Fusionar BROKEN con RECOVERING (ambos representan anomalia)
#   B) Eliminar BROKEN y tratar como problema binario NORMAL vs RECOVERING
# Justificar la decision en la celda Markdown siguiente

# Paso 3: Codificar etiquetas como enteros
# Ejemplo si se elige opcion A:
#   NORMAL     -> 0
#   RECOVERING -> 1
#   BROKEN     -> 1  (fusionado con RECOVERING)

# df['estado_cod'] = df['machine_status'].map({'NORMAL': 0, 'RECOVERING': 1, 'BROKEN': ???})
# print(f"\nDistribucion final de clases:")
# print(df['estado_cod'].value_counts())

# Guardar el numero de clases para usar en el modelo
# N_CLASES = df['estado_cod'].nunique()
# print(f"Numero de clases: {N_CLASES}")

print("[COMPLETAR] Implementar la decision sobre BROKEN y la codificacion.")

### Respuesta escrita 1.3 — Decision sobre la clase BROKEN

**[COMPLETAR — hacer doble clic para editar]**

Opcion elegida: *(A o B)*

Justificacion:

*(Escribir aqui. Minimo 3 lineas. Argumentar por que la clase BROKEN no puede usarse directamente para entrenamiento y que consecuencia tiene esa decision sobre el tipo de problema — binario o multiclase.)*

### Respuesta escrita 1.4 — Por que se necesita modelado secuencial

**[COMPLETAR — hacer doble clic para editar]**

*(Explicar por que un clasificador clasico — por ejemplo Random Forest aplicado fila por fila — no puede detectar el estado RECOVERING de forma anticipada. Usar conceptos de dependencia temporal y contexto historico. Minimo 4 lineas.)*

---
## FASE 2 — Implementacion de modelos
**Responsables:** Integrantes 1, 2 y 3  
**Puntaje:** 9 puntos  
**Tiempo estimado:** 60 minutos

### Rubrica

| Criterio | Puntos |
|----------|--------|
| Construccion de ventanas temporales sin data leakage | 2.0 |
| Modelo RNN funcional con curvas de entrenamiento | 2.0 |
| Modelo LSTM o GRU funcional | 2.0 |
| Comparacion de metricas con enfasis en clase anomalia | 2.0 |
| Justificacion tecnica LSTM vs GRU | 1.0 |
| **Total Fase 2** | **9.0** |

In [ ]:
# ============================================================
# FASE 2 — Bloque A: Construccion de ventanas temporales
# Responsable: Integrante 1
# [COMPLETAR]
# ============================================================

# Una ventana temporal toma los ultimos LOOKBACK pasos de tiempo
# para predecir el estado en el instante siguiente.
# Ejemplo con LOOKBACK=3:
#   X[i] = [mediciones en t-3, t-2, t-1]  ->  y[i] = estado en t

LOOKBACK = 30   # usar los ultimos 30 minutos de datos

# Paso 1: Normalizar los sensores
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(df[sensores_validos])
# y_array  = df['estado_cod'].values

# Paso 2: Funcion para construir secuencias (X de forma [n, lookback, n_features])
def crear_secuencias(X, y, lookback):
    """
    Convierte arrays planos en secuencias para modelos recurrentes.

    Parametros:
        X        : array normalizado de shape (n_muestras, n_features)
        y        : array de etiquetas de shape (n_muestras,)
        lookback : numero de pasos de tiempo en cada ventana

    Retorna:
        X_seq : array de shape (n_muestras - lookback, lookback, n_features)
        y_seq : array de shape (n_muestras - lookback,)
    """
    X_seq, y_seq = [], []                      # listas vacias para acumular ventanas
    for i in range(lookback, len(X)):          # iterar desde lookback hasta el final
        X_seq.append(X[i - lookback : i])      # ventana de lookback pasos anteriores
        y_seq.append(y[i])                     # etiqueta del paso actual
    return np.array(X_seq), np.array(y_seq)    # convertir listas a arrays numpy

# Paso 3: Dividir en train y test ANTES de construir las ventanas
# CRITICO: en series temporales NO se hace shuffle
# Usar los primeros 80% como train y los ultimos 20% como test
# Esto evita data leakage (el modelo no ve el futuro durante el entrenamiento)

# TODO:
# corte = int(len(X_scaled) * 0.80)
# X_train_sc, X_test_sc = X_scaled[:corte], X_scaled[corte:]
# y_train_arr, y_test_arr = y_array[:corte], y_array[corte:]

# X_train, y_train = crear_secuencias(X_train_sc, y_train_arr, LOOKBACK)
# X_test,  y_test  = crear_secuencias(X_test_sc,  y_test_arr,  LOOKBACK)

# print(f"X_train shape: {X_train.shape}  <- (muestras, lookback, features)")
# print(f"X_test  shape: {X_test.shape}")
# print(f"y_train shape: {y_train.shape}")
# print(f"y_test  shape: {y_test.shape}")

# Paso 4: Calcular pesos de clase para compensar desbalance
# El estado RECOVERING es mucho menos frecuente que NORMAL
# Los pesos penalizan mas los errores en la clase minoritaria
# clases = np.unique(y_train)
# pesos  = compute_class_weight('balanced', classes=clases, y=y_train)
# class_weight_dict = dict(zip(clases, pesos))
# print(f"\nPesos de clase: {class_weight_dict}")

print("[COMPLETAR] Implementar los pasos de construccion de ventanas.")

In [ ]:
# ============================================================
# FASE 2 — Bloque B: Modelo 1 — RNN Simple
# Responsable: Integrante 2
# [COMPLETAR]
# ============================================================

# Construir una RNN simple con la siguiente arquitectura minima:
#   - Capa SimpleRNN con 64 unidades y return_sequences=False
#   - Capa Dropout(0.2) para regularizacion
#   - Capa Dense de salida con activacion adecuada al numero de clases
#     (sigmoid si binario, softmax si multiclase)

# Parametros recomendados de entrenamiento:
EPOCHS     = 20    # maximo de epocas
BATCH_SIZE = 64    # tamanio del lote

# EarlyStopping: detener si val_loss no mejora en 5 epocas consecutivas
early_stop = EarlyStopping(
    monitor='val_loss',    # metrica a monitorear
    patience=5,            # epocas de tolerancia sin mejora
    restore_best_weights=True  # recuperar los pesos de la mejor epoca
)

# TODO: construir, compilar y entrenar el modelo RNN
# modelo_rnn = Sequential([
#     SimpleRNN(...),
#     Dropout(...),
#     Dense(...)
# ])
# modelo_rnn.compile(
#     optimizer='adam',
#     loss=...,          # 'binary_crossentropy' o 'sparse_categorical_crossentropy'
#     metrics=['accuracy']
# )
# modelo_rnn.summary()

# historia_rnn = modelo_rnn.fit(
#     X_train, y_train,
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     validation_split=0.15,
#     class_weight=class_weight_dict,
#     callbacks=[early_stop],
#     verbose=1
# )

print("[COMPLETAR] Construir, compilar y entrenar el modelo RNN.")

In [ ]:
# ============================================================
# FASE 2 — Bloque C: Modelo 2 — LSTM o GRU
# Responsable: Integrante 3
# [COMPLETAR]
# ============================================================

# El grupo elige LSTM o GRU y justifica la decision en la celda
# Markdown siguiente.

# Arquitectura minima:
#   - Primera capa LSTM o GRU con 64 unidades, return_sequences=True
#   - Segunda capa LSTM o GRU con 32 unidades, return_sequences=False
#   - Capa Dropout(0.3)
#   - Capa Dense de salida

# TODO: construir, compilar y entrenar el modelo LSTM/GRU
# modelo_avanzado = Sequential([
#     ...
# ])
# modelo_avanzado.compile(...)
# modelo_avanzado.summary()

# historia_avanzado = modelo_avanzado.fit(
#     X_train, y_train,
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     validation_split=0.15,
#     class_weight=class_weight_dict,
#     callbacks=[early_stop],
#     verbose=1
# )

print("[COMPLETAR] Construir, compilar y entrenar el modelo LSTM o GRU.")

### Respuesta escrita 2.1 — Justificacion de la eleccion LSTM vs GRU

**[COMPLETAR — hacer doble clic para editar]**

Arquitectura elegida: *(LSTM / GRU)*

**Si se eligio LSTM:** explicar como las tres compuertas (forget, input, output) permiten al modelo retener informacion relevante durante secuencias largas y por que eso es importante para detectar RECOVERING con anticipacion de 30 o mas pasos.

**Si se eligio GRU:** explicar por que dos compuertas (reset, update) son suficientes para este caso, y en que condiciones la GRU es preferible a la LSTM en terminos de velocidad de entrenamiento y numero de parametros.

*(Respuesta minima: 5 lineas con argumento tecnico, no solo definicion.)*

In [ ]:
# ============================================================
# FASE 2 — Bloque D: Comparacion de modelos
# Responsable: Integrante 3 con revision de Integrante 2
# [COMPLETAR]
# ============================================================

# --- Grafico 1: Curvas de loss de entrenamiento ---
# Graficar en la misma figura:
#   - loss de train y validation del modelo RNN
#   - loss de train y validation del modelo LSTM/GRU
# Usar colores distintos y leyenda clara

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
# fig.suptitle('Comparacion de curvas de entrenamiento')

# ax1.plot(historia_rnn.history['loss'],     label='RNN train',     color='steelblue')
# ax1.plot(historia_rnn.history['val_loss'], label='RNN val',       color='steelblue', linestyle='--')
# ax1.set_xlabel('Epoca'); ax1.set_ylabel('Loss')
# ax1.set_title('RNN Simple'); ax1.legend(); ax1.grid(True, alpha=0.3)

# ax2.plot(historia_avanzado.history['loss'],     label='LSTM/GRU train', color='darkorange')
# ax2.plot(historia_avanzado.history['val_loss'], label='LSTM/GRU val',   color='darkorange', linestyle='--')
# ax2.set_xlabel('Epoca'); ax2.set_ylabel('Loss')
# ax2.set_title('LSTM / GRU'); ax2.legend(); ax2.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

# --- Grafico 2: Prediccion vs realidad sobre el test set ---
# Graficar las primeras 500 predicciones del mejor modelo
# comparadas con las etiquetas reales

# TODO: predecir sobre X_test con ambos modelos
# y_pred_rnn      = ...
# y_pred_avanzado = ...

# --- Metricas: classification_report para ambos modelos ---
# Prestar especial atencion al recall de la clase RECOVERING/anomalia
# Un recall bajo significa que el modelo no detecta las fallas a tiempo

# NOMBRES_CLASES = ['Normal', 'Anomalia']   # ajustar segun decision de Fase 1

# print("=" * 50)
# print("MODELO RNN SIMPLE")
# print("=" * 50)
# print(classification_report(y_test, y_pred_rnn, target_names=NOMBRES_CLASES))

# print("=" * 50)
# print("MODELO LSTM / GRU")
# print("=" * 50)
# print(classification_report(y_test, y_pred_avanzado, target_names=NOMBRES_CLASES))

print("[COMPLETAR] Implementar graficos y metricas comparativas.")

### Respuesta escrita 2.2 — Interpretacion de metricas

**[COMPLETAR — hacer doble clic para editar]**

Completar la siguiente tabla con los valores obtenidos:

| Metrica | RNN Simple | LSTM/GRU | Diferencia |
|---------|-----------|----------|------------|
| Accuracy global | | | |
| Precision clase anomalia | | | |
| Recall clase anomalia | | | |
| F1-score clase anomalia | | | |

Interpretacion (minimo 3 lineas):

*(Explicar que metrica es mas importante para el problema de mantenimiento predictivo y por que. Discutir el trade-off entre precision y recall en el contexto operativo: que es peor, una falsa alarma o no detectar una falla real?)*

---
## FASE 3 — Analisis de gradientes
**Responsable principal:** Integrante 2  
**Puntaje:** 3 puntos  
**Tiempo estimado:** 10 minutos

Esta fase no requiere codigo. Las respuestas se escriben en las celdas Markdown siguientes.

### Rubrica

| Criterio | Puntos |
|----------|--------|
| Explicacion correcta del vanishing gradient con BPTT | 1.5 |
| Propuesta de LOOKBACK alternativo con argumento valido | 1.5 |
| **Total Fase 3** | **3.0** |

### Pregunta 3.1 — Vanishing Gradient en BPTT

**[COMPLETAR — hacer doble clic para editar]**

El gradiente de la perdida respecto a los pesos de una RNN en el paso $t$ se calcula propagando hacia atras a traves del tiempo:

$$\frac{\partial L}{\partial W} = \sum_{t} \frac{\partial L_t}{\partial h_t} \prod_{k=t+1}^{T} \frac{\partial h_k}{\partial h_{k-1}}$$

Responder:

**a)** Que ocurre con el producto $\prod_{k=t+1}^{T} \frac{\partial h_k}{\partial h_{k-1}}$ cuando $\left\|\frac{\partial h_k}{\partial h_{k-1}}\right\| < 1$ para muchos pasos $k$?

*(Respuesta aqui)*

**b)** Que consecuencia practica tiene ese comportamiento sobre la capacidad de la RNN para detectar el estado RECOVERING cuando este precede a la falla por mas de 20 pasos de tiempo?

*(Respuesta aqui)*

**c)** Como resuelve la arquitectura elegida en Fase 2 (LSTM o GRU) este problema? Describir el mecanismo concreto, no solo nombrarlo.

*(Respuesta aqui)*

### Pregunta 3.2 — Seleccion del LOOKBACK optimo

**[COMPLETAR — hacer doble clic para editar]**

En Fase 2 se uso `LOOKBACK = 30` minutos. A partir de los graficos de series temporales generados en Fase 1:

**a)** Proponer un valor alternativo de LOOKBACK (puede ser mayor o menor) y argumentar por que ese valor podria mejorar la deteccion del estado RECOVERING. Apoyarse en lo observado visualmente en las series.

*(Respuesta aqui)*

**b)** Que riesgo computacional existe al aumentar excesivamente el LOOKBACK en una RNN simple?

*(Respuesta aqui)*

---
## FASE 4 — Presentacion ejecutiva
**Responsable principal:** Integrante 4  
**Puntaje:** 4 puntos  
**Tiempo estimado:** 10 minutos de preparacion + 3 minutos de exposicion oral

### Rubrica

| Criterio | Puntos |
|----------|--------|
| Claridad: ausencia de jerga tecnica innecesaria | 1.5 |
| Precision: los datos del notebook se reflejan correctamente | 1.5 |
| Coherencia entre resultados y recomendacion operativa | 1.0 |
| **Total Fase 4** | **4.0** |

### Contenido requerido por diapositiva

Preparar 4 diapositivas en Google Slides, PowerPoint o Canva, dirigidas al **gerente de planta** (no al equipo tecnico). Compartir el enlace o adjuntar el archivo al notebook.

| Diapositiva | Titulo sugerido | Contenido obligatorio |
|-------------|-----------------|----------------------|
| 1 | El problema en numeros | Frecuencia de fallas en el dataset; costo estimado de parada correctiva vs preventiva (el grupo puede asumir un costo por hora de parada) |
| 2 | Como funciona el modelo | Analogia visual del lookback; que señal de los sensores detecta antes de la falla |
| 3 | Que tan confiable es | Recall de la clase anomalia en lenguaje llano; grafico de prediccion vs estado real en un episodio del test set |
| 4 | Que falta antes de produccion | Al menos 3 limitaciones concretas; propuesta de siguiente paso |

### Enlace a las diapositivas

**[COMPLETAR — hacer doble clic para editar]**

Enlace a las diapositivas:

*(Pegar el enlace de Google Slides o indicar el nombre del archivo adjunto)*

---
## BONUS — Dashboard interactivo con Streamlit
**Puntaje adicional:** 2 puntos (fuera del tiempo del examen)  
**Plazo de entrega:** 48 horas despues del examen

Implementar el archivo `dashboard_bomba.py` usando la plantilla base proporcionada a continuacion. El grupo debe reemplazar los placeholders con los resultados reales obtenidos en Fase 2.

**Criterio de evaluacion del bonus:**
- El dashboard corre sin errores y muestra datos reales del modelo: 1 pt
- El semaforo de alerta cambia correctamente segun el umbral del slider: 1 pt

In [ ]:
# ============================================================
# BONUS — Plantilla base del dashboard
# Guardar este codigo como dashboard_bomba.py
# Ejecutar con: streamlit run dashboard_bomba.py
# ============================================================

# Este bloque solo muestra la plantilla. No ejecutar como celda de Colab.
# Copiar el contenido a un archivo .py separado.

PLANTILLA_DASHBOARD = '''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Configuracion de la pagina
st.set_page_config(
    page_title="Monitor de Bomba — Mantenimiento Predictivo",
    layout="wide"
)

st.title("Sistema de Alerta Temprana — Bomba Industrial")
st.caption("Modelo LSTM/GRU entrenado sobre datos reales (Kaggle: pump_sensor_data)")

# ---- Sidebar ----
st.sidebar.header("Configuracion del operador")
umbral_alerta = st.sidebar.slider(
    "Umbral de probabilidad para alerta",
    min_value=0.30, max_value=0.90, value=0.50, step=0.05
)
ventana_viz = st.sidebar.slider(
    "Ventana de visualizacion (pasos)",
    min_value=100, max_value=2000, value=500
)

# ---- REEMPLAZAR: cargar resultados reales del notebook ----
# y_true_test        -> array con etiquetas reales del test set
# y_pred_test        -> array con predicciones del mejor modelo
# y_prob_test        -> array de shape (n, n_clases) con probabilidades
# timestamps_test    -> array de timestamps del test set
# recall_anomalia    -> float, recall de la clase anomalia
#
# Ejemplo de carga si se exportaron desde el notebook:
# import pickle
# with open('resultados.pkl', 'rb') as f:
#     resultados = pickle.load(f)
# y_true_test     = resultados['y_true']
# y_pred_test     = resultados['y_pred']
# y_prob_test     = resultados['y_prob']
# timestamps_test = resultados['timestamps']
# recall_anomalia = resultados['recall']

# Placeholders de demostración (reemplazar con datos reales)
n = ventana_viz
y_true_test     = np.random.choice([0, 1], size=n, p=[0.90, 0.10])
y_prob_test     = np.column_stack([1 - y_true_test * 0.8,
                                    y_true_test * 0.8])
y_pred_test     = (y_prob_test[:, 1] >= umbral_alerta).astype(int)
timestamps_test = pd.date_range("2018-04-01", periods=n, freq="1min")
recall_anomalia = 0.74   # reemplazar con valor real

# ---- Indicadores principales ----
ultimo_estado = "ANOMALIA" if y_pred_test[-1] == 1 else "NORMAL"
ultima_prob   = y_prob_test[-1, 1]
n_alertas     = (y_pred_test == 1).sum()

col1, col2, col3 = st.columns(3)

with col1:
    color = "#d32f2f" if ultimo_estado == "ANOMALIA" else "#388e3c"
    st.markdown(
        f"<div style='background:{color};padding:18px;border-radius:8px;"
        f"text-align:center;color:white;font-size:18px;font-weight:bold;'>"
        f"Estado actual: {ultimo_estado}</div>",
        unsafe_allow_html=True
    )

with col2:
    st.metric("Probabilidad de anomalia", f"{ultima_prob:.1%}")
    st.progress(float(ultima_prob))

with col3:
    st.metric(
        "Recall del modelo", f"{recall_anomalia:.1%}",
        help="De cada 10 fallas reales, el modelo detecta este porcentaje"
    )
    st.metric("Alertas generadas", n_alertas)

st.divider()

# ---- Grafico principal ----
st.subheader("Prediccion del modelo vs estado real")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=timestamps_test, y=y_true_test,
    mode="lines", name="Estado real",
    line=dict(color="steelblue", width=1.5)
))
fig.add_trace(go.Scatter(
    x=timestamps_test, y=y_pred_test,
    mode="lines", name="Prediccion modelo",
    line=dict(color="darkorange", width=1.5, dash="dot")
))
fig.add_trace(go.Scatter(
    x=timestamps_test, y=y_prob_test[:, 1],
    mode="lines", name="Probabilidad anomalia",
    line=dict(color="red", width=1, dash="dash"),
    opacity=0.5
))
fig.add_hline(
    y=umbral_alerta, line_dash="dot",
    line_color="gray", annotation_text=f"Umbral: {umbral_alerta:.2f}"
)
fig.update_layout(
    xaxis_title="Tiempo",
    yaxis=dict(tickvals=[0, 1], ticktext=["NORMAL", "ANOMALIA"]),
    legend=dict(orientation="h", y=-0.2),
    height=350,
    margin=dict(l=40, r=20, t=20, b=20)
)
st.plotly_chart(fig, use_container_width=True)

st.caption("Prototipo academico. No usar en produccion sin validacion adicional.")
'''

# Guardar la plantilla como archivo .py para uso fuera de Colab
with open('dashboard_bomba.py', 'w') as f:   # abrir archivo en modo escritura
    f.write(PLANTILLA_DASHBOARD)              # escribir el contenido de la plantilla

print("Plantilla guardada como dashboard_bomba.py")
print("Descargar el archivo y ejecutar con: streamlit run dashboard_bomba.py")
print("Reemplazar los placeholders con los resultados reales del notebook.")

---
## Resumen de puntaje

**[Para uso del docente — no modificar]**

| Fase | Criterio | Puntaje maximo | Puntaje obtenido |
|------|----------|:--------------:|:----------------:|
| 1 | Seleccion de sensores y NaN | 1.0 | |
| 1 | Grafico de series temporales | 1.0 | |
| 1 | Decision sobre BROKEN | 1.0 | |
| 1 | Justificacion modelado secuencial | 1.0 | |
| 2 | Ventanas temporales sin data leakage | 2.0 | |
| 2 | Modelo RNN funcional | 2.0 | |
| 2 | Modelo LSTM/GRU funcional | 2.0 | |
| 2 | Comparacion de metricas | 2.0 | |
| 2 | Justificacion LSTM vs GRU | 1.0 | |
| 3 | Vanishing gradient (BPTT) | 1.5 | |
| 3 | LOOKBACK optimo | 1.5 | |
| 4 | Claridad de comunicacion | 1.5 | |
| 4 | Precision de contenidos | 1.5 | |
| 4 | Coherencia resultados-recomendacion | 1.0 | |
| **Total** | | **20.0** | |
| Bonus | Dashboard Streamlit | +2.0 | |

---

